In [ ]:
import numpy as np
from benchmarker import KeplerBenchmarker



M = np.deg2rad(7)
e = 0.999

E01 = M
E02 = M + (e * np.sin(M)) / (1 - np.sin(M + e) + np.sin(M))
E03 = M + e * np.sin(M + e* np.sin(M + e))

# E0 = np.deg2rad(38.52700657)
B= 1.173439404
A = -0.584013113
C = 0.809460441
D = 0.077357763

a = (B * np.sin(M) + D * np.cos(M)) / ( (1 / e) - A * np.sin(M) - C * np.cos(M) )
E04 = M  + e * (np.sin( M + e * np.sin(M + a)))







In [32]:
print(E01, E02, E03, E04)

0.12217304763960307 0.6724231156516713 0.9744121394497225 0.9223461083933904


In [7]:
print("===== E01 =====")
print(f"E0:={E01} | M={M} | e={e}")
r1=KeplerBenchmarker(M, e, E01).run_benchmark()
r1


===== E01 =====
E0:=0.12217304763960307 | M=0.12217304763960307 | e=0.999


,Method,Root (Rad),Iterations,Avg (μs),Std Dev (μs),Best (μs),Residual Error
1,Halley,0.912288,6,7.3608,0.0778,7.2603,5.55e-17
0,Newton,0.912288,44,31.9312,0.9098,31.2457,4.27e-12


In [8]:
print("\n===== E02 =====")
print(f"E0:={E02} | M={M} | e={e}")
r2=KeplerBenchmarker(M, e, E02).run_benchmark()
r2



===== E02 =====
E0:=0.6724231156516713 | M=0.12217304763960307 | e=0.999


,Method,Root (Rad),Iterations,Avg (μs),Std Dev (μs),Best (μs),Residual Error
1,Halley,0.912288,3,3.6617,0.0108,3.6487,5.55e-17
0,Newton,0.912288,5,3.6799,0.0479,3.6257,5.55e-17


In [9]:
print("\n===== E03 =====")
print(f"E0:={E03} | M={M} | e={e}")
r3=KeplerBenchmarker(M, e, E03).run_benchmark()
r3


===== E03 =====
E0:=0.9744121394497225 | M=0.12217304763960307 | e=0.999


,Method,Root (Rad),Iterations,Avg (μs),Std Dev (μs),Best (μs),Residual Error
1,Halley,0.912288,3,3.7319,0.1094,3.6513,5.55e-17
0,Newton,0.912288,4,2.9036,0.1139,2.8347,5.55e-17


In [10]:
print("\n===== E04 =====")
print(f"E0:={E04} | M={M} | e={e}")
r4=KeplerBenchmarker(M, e, E04).run_benchmark()
r4


===== E04 =====
E0:=0.9223461083933904 | M=0.12217304763960307 | e=0.999


,Method,Root (Rad),Iterations,Avg (μs),Std Dev (μs),Best (μs),Residual Error
1,Halley,0.912288,2,2.6143,0.0068,2.6077,5.55e-17
0,Newton,0.912288,3,2.3289,0.3542,2.1814,5.55e-17


In [31]:
import pandas as pd
import numpy as np

# Assuming you have df1, df2, df3, df4
dfs = [r1, r2, r3, r4]
cases = ['Case 1', 'Case 2', 'Case 3', 'Case 4']

for i, df in enumerate(dfs):
    df['Case'] = cases[i]

# Combine only Newton and Halley for this specific analysis
combined_df = pd.concat(dfs, ignore_index=True)
combined_df = combined_df[combined_df['Method'].isin(['Newton', 'Halley'])]

# Define the columns we are weighting
criteria = ['Iterations', 'Avg (μs)', 'Std Dev (μs)']

# Normalize: Lower values should result in a higher score (closer to 1.0)
for col in criteria:
    min_val = combined_df[col].min()
    combined_df[f'norm_{col}'] = min_val / combined_df[col]


weights = {'norm_Iterations': 1/3, 'norm_Avg (μs)': 1/3, 'norm_Std Dev (μs)': 1/3}

# Calculate the final Score
combined_df['SAW_Score'] = (
    combined_df['norm_Iterations'] * weights['norm_Iterations'] +
    combined_df['norm_Avg (μs)'] * weights['norm_Avg (μs)'] +
    combined_df['norm_Std Dev (μs)'] * weights['norm_Std Dev (μs)']
)


# Group by Method to see the overall winner
final_ranking = combined_df.groupby('Method')['SAW_Score'].mean().sort_values(ascending=False)

print("--- Final SAW Results (Higher is Better) ---")
print(final_ranking)

--- Final SAW Results (Higher is Better) ---
Method
Halley    0.576096
Newton    0.362361
Name: SAW_Score, dtype: float64


In [23]:
combined_df.groupby('Method').head()

,Method,Root (Rad),Iterations,Avg (μs),Std Dev (μs),Best (μs),Residual Error,Case,norm_Iterations,norm_Avg (μs),norm_Std Dev (μs),SAW_Score
0,Halley,0.912288,6,7.3608,0.0778,7.2603,5.55e-17,Case 1,0.333333,0.316392,0.087404,0.245710
1,Newton,0.912288,44,31.9312,0.9098,31.2457,4.27e-12,Case 1,0.045455,0.072935,0.007474,0.041955
2,Halley,0.912288,3,3.6617,0.0108,3.6487,5.55e-17,Case 2,0.666667,0.636016,0.629630,0.644104
3,Newton,0.912288,5,3.6799,0.0479,3.6257,5.55e-17,Case 2,0.400000,0.632870,0.141962,0.391611
4,Halley,0.912288,3,3.7319,0.1094,3.6513,5.55e-17,Case 3,0.666667,0.624052,0.062157,0.450959
5,Newton,0.912288,4,2.9036,0.1139,2.8347,5.55e-17,Case 3,0.500000,0.802073,0.059701,0.453925
6,Halley,0.912288,2,2.6143,0.0068,2.6077,5.55e-17,Case 4,1.000000,0.890831,1.000000,0.963610
7,Newton,0.912288,3,2.3289,0.3542,2.1814,5.55e-17,Case 4,0.666667,1.000000,0.019198,0.561955
